# 拓扑码的神经解码器：Torlai & Melko (2017) 复现

G. Torlai and R. G. Melko, *Neural Decoder for Topological Codes*, **Phys. Rev. Lett. 119**, 030501 (2017), [doi:10.1103/PhysRevLett.119.030501](https://doi.org/10.1103/PhysRevLett.119.030501)。论文 PDF：`paper/pdf/A Neural Decoder for Topological Codes.pdf`。

本 Notebook 用 AI-for-QEC 标准工作流复现论文：toric code、相位翻转 code-capacity 噪声、联合 `[e | S]` 受限玻尔兹曼机（RBM）、CD-k 训练、syndrome 钳制的 Gibbs 解码（算法 1），并在同一 test split 上与最小权重完美匹配（MWPM）比较。Notebook 只负责配置、编排顺序和展示；数据生成、训练、解码、评估、Artifact 与恢复都由 `ai_qec` 的本地 runtime 实现，并通过 `ai_qec.notebook_api` 调用。

| 论文 | 工作流 Stage | 实现（经 Registry 选择） |
|---|---|---|
| 算法 1 第 1–2 行：`e₀`、`S₀ = S(e₀)` | `resolve_dataset` | `stim-syndrome-cpu`：Stim 帧模拟器采样 Z 错误、顶点 syndrome 与逻辑可观测量；提交前用 vendor-neutral toric code 逐样本校验 |
| 第 3 行：训练 `RBM = {e, S, h}` | `train` | `pytorch` trainer + 模型无关的 `pytorch-cuda-graph` step executor + `contrastive-divergence` + `sgd`；每个 epoch 写恢复 checkpoint |
| 第 3–8 行：钳制 `S₀`，块 Gibbs 采样直到 `S(e) = S₀`，`r = e` | `evaluate_accuracy` | `joint-error-syndrome-rbm` 的 Gibbs decoder，全部 test 样本在 GPU 上并行成链 |
| MWPM 对照（曼哈顿距离） | `evaluate_accuracy` | `pymatching-cpu-decoder`，均匀边权 |
| 图 3、图 4 | `visualize` 与本 Notebook 第 4–5 节 | `qec.plot_failure_rate_sweep`、`qec.plot_logical_class_histograms` |

**运行方式。** 内核环境需要以 editable 方式安装本项目及运行时依赖（`pip install -e ".[runtime]"`）。带 `run-experiment` 标签的 cells 才会创建 runtime 并执行实验；其余 cells 只做定义，可以安全地单独执行。`paper` profile 训练 22 个网格点，在本机（RTX 4060 Laptop GPU）首次约需 35 分钟；之后再 Run All 会为每个点创建新的 Attempt，并复用已校验的数据集、模型、评估和性能结果，几秒内完成。`smoke` profile 只跑 3 个小点，结果不能作为科学结论。中断（包括内核重启）后再次执行，会从最近一个校验通过的 epoch checkpoint 继续训练。

In [ ]:
import ai_qec.notebook_api as qec
PROJECT_ROOT = qec.find_project_root()

## 1. 实验配置

`CONFIG` 是一个 (L, p) 点的完整配置，所有可选实现都用 Registry key 表达；`qec.LocalNotebookPlatform.create_experiment` 会在创建任何目录之前完成预检（`unresolved`、未知键、未登记实现、Stim 版本、CUDA 可用性、噪声能否精确编译）。`qec.config_grid` 从它展开网格：轴（L、p）、按 L 耦合的超参数和 smoke 的共享覆盖都显式声明，覆盖只能指向 `CONFIG` 中已有的字段，拼错或重复覆盖会直接报错。每个点是独立的 Experiment（身份为完整配置的摘要）。只改 Gate 时 Experiment 身份会变，但训练、科学评估与性能评估会按 Stage 复用键从已有 Experiment 复用，不会重新训练。

与论文设置的差异（均为显式选择）：

- 论文对每个 p 分别网格搜索超参数；这里按 L 固定一组：L=4 用 64 个隐藏单元、30 epoch，L=6 用 128 个隐藏单元、40 epoch；二者都用 SGD lr=0.1、CD-10、batch 100、10⁵ 个训练样本。取值来自本仓库的原型扫描；动量 0.9 会先快速收敛再退化，因此保持论文的朴素 SGD。
- 论文的 Gibbs 截止步数未给出数值；这里 burn-in 100 步（论文：`N_eq ∝ 10²`），`max_steps = 20000`，超时样本计为逻辑失败并单独计数。
- MWPM 使用 PyMatching（稀疏 blossom，精确最小权重），边权均匀，即论文中的曼哈顿距离。
- Accuracy Gate 是本项目的约定而非论文内容：在同一 test shots 上做配对相对非劣效比较，当 `LER_RBM − 1.15·LER_MWPM` 的 95% 配对区间（MOVER，Wilson 分量）上界不超过 0 时判定 PASS，即 RBM 的失败率最多比 MWPM 高 15%。PASS 之后才测量解码吞吐量。容差最初登记为绝对值 0.02，看到结果后于 2026-09-22 改为相对 15%：绝对容差在 p=0.05 相当于允许约 50% 的相对差距，在 p=0.15 只允许约 4%，严格程度随 p 变化。0.02 下的 22 个 Experiment 及其判定保留在 `runs/`。

In [ ]:
CONFIG: dict[str, object] = {
    "schema_version": "0.1",
    "technology_profile": "v0-1",
    "experiment": {"name": "torlai-melko-2017", "master_seed": 2017},
    "qec": {
        "code_family": "toric",
        "distance": 4,
        "rounds": 1,
        "logical_basis": "X",
        "circuit_family": "code-capacity",
    },
    "noise": {
        "family": "independent-phase-flip",
        "adapter": "stim-noise",
        "parameters": {"physical_error_rate": 0.08},
    },
    "dataset": {
        "generator": "stim-syndrome-cpu",
        "generator_version": "1.16.0",
        "backend_semantics": "exact",
        "train_samples": 100_000,
        "validation_samples": 10_000,
        "test_samples": 10_000,
        "seed": 2017,
        "split_policy": "independent-streams",
        "schema_version": "qec-batch-v1",
    },
    "model": {
        "family": "joint-error-syndrome-rbm",
        "architecture_version": "1",
        "parameters": {"hidden_units": 64, "init_std": 0.01},
        "decoding": {"burn_in": 100, "max_steps": 20_000},
    },
    "training": {
        "optimizer": "sgd",
        "optimizer_parameters": {"weight_decay": 0.0, "momentum": 0.0},
        "learning_rate": 0.1,
        "epochs": 30,
        "batch_size": 100,
        "scheduler": "constant",
        "loss": "contrastive-divergence",
        "loss_parameters": {"cd_steps": 10},
    },
    "execution": {
        "trainer_framework": "pytorch",
        "device": "cuda",
        "cpu_count": 1,
        "gpu_count": 1,
        "distributed": False,
        "num_workers": 0,
        "mixed_precision": False,
        "compile_model": False,
        "step_executor": "pytorch-cuda-graph",
        "step_executor_options": {"max_graphs": 2},
    },
    "data_pipeline": {
        "cpu_to_gpu": {"technology": "pytorch-dataloader-h2d", "pin_memory": True, "non_blocking": True},
    },
    "scientific_evaluation": {
        "baseline_decoders": ["pymatching-cpu-decoder"],
        "primary_metric": "logical_error_rate",
        "confidence_level": 0.95,
        "stopping_rule": "fixed-shots",
        "invalid_sample_policy": "count-as-failure",
    },
    "accuracy_gate": {
        "gate_id": "gate-a",
        "baseline_decoder": "pymatching-cpu-decoder",
        "primary_metric": "logical_error_rate",
        "comparison_rule": "paired-relative-non-inferiority",
        "tolerance": 0.15,
        "confidence_level": 0.95,
    },
    "performance": {"shots": 1000, "repetitions": 3, "warmup": 1},
}

# 论文图 3 的网格：L ∈ {4, 6}，p = 0.05, 0.06, …, 0.15。论文逐 p 搜索超参数，这里按 L 固定。
PAPER_ERROR_RATES = tuple(round(0.05 + 0.01 * step, 2) for step in range(11))
GRIDS = {
    "paper": qec.config_grid(
        CONFIG,
        name="torlai-melko-2017-paper-l{L}-p{p:.2f}",
        axes={
            "L": ("qec.distance", (4, 6)),
            "p": ("noise.parameters.physical_error_rate", PAPER_ERROR_RATES),
        },
        coupled={
            "L": {
                4: {"model.parameters.hidden_units": 64, "training.epochs": 30},
                6: {"model.parameters.hidden_units": 128, "training.epochs": 40},
            },
        },
    ),
    "smoke": qec.config_grid(
        CONFIG,
        name="torlai-melko-2017-smoke-l{L}-p{p:.2f}",
        axes={
            "L": ("qec.distance", (4,)),
            "p": ("noise.parameters.physical_error_rate", (0.05, 0.10, 0.15)),
        },
        overrides={
            "dataset.train_samples": 20_000,
            "dataset.validation_samples": 2_000,
            "dataset.test_samples": 2_000,
            "training.epochs": 10,
        },
    ),
}
PROFILE = "paper"
GRID = GRIDS[PROFILE]
len(GRID)

## 2. 标准编排

`run_paper` 保持平台的标准研究顺序：Start/Recover → Resolve Dataset → Train → Scientific Evaluation → Accuracy Gate → 仅在 PASS 时 Performance → Visualize → Finish。runtime 通过参数注入；任何满足 `qec.NotebookPlatform` 的实现都可以替换本地 runtime。baseline 列表取自预注册的配置，保证评估与 Gate 使用同一个 baseline。

In [ ]:
from collections.abc import Mapping
from typing import Any


def run_paper(
    runtime: qec.NotebookPlatform,
    config: Mapping[str, Any] = CONFIG,
) -> tuple[
    qec.ScientificEvaluationResult,
    qec.ScientificAcceptanceResult,
    qec.ArtifactRef | None,
    tuple[qec.ArtifactRef, ...],
]:
    experiment = runtime.create_experiment(config)
    run = experiment.start_or_recover()

    dataset = run.resolve_dataset()
    model = run.train(dataset)

    scientific_result = run.evaluate_accuracy(
        model=model,
        dataset=dataset,
        baselines=tuple(config["scientific_evaluation"]["baseline_decoders"]),
    )
    acceptance = run.check_accuracy_gate(scientific_result)

    performance_result = None
    if acceptance.decision is qec.GateDecision.PASS:
        performance_result = run.evaluate_performance(
            model=model,
            dataset=dataset,
        )

    figures = run.visualize(
        scientific_result=scientific_result,
        acceptance=acceptance,
        performance_result=performance_result,
    )
    run.finish()
    return scientific_result, acceptance, performance_result, figures

## 3. 构造本地 runtime 并执行网格

`LocalNotebookPlatform` 把数据集写入 `datasets/`（按 DatasetKey 复用，命中前完整校验 checksum），把每个 Experiment 的 Attempt、Stage 记录、checkpoint、预测、指标和图写入 `runs/`；两个目录都被 git 忽略。每个 Stage 结束时打印一行摘要。

In [ ]:
RUNTIME = qec.LocalNotebookPlatform(PROJECT_ROOT)
RESULTS = {(point.values["L"], point.values["p"]): run_paper(RUNTIME, point.config) for point in GRID}

POINTS = [
    qec.SweepPoint(distance=distance, physical_error_rate=rate, scientific=result[0], acceptance=result[1])
    for (distance, rate), result in RESULTS.items()
]

In [ ]:
# 每个网格点的 P_fail（Wilson 95% 区间）、RBM 超时数与 Gate 判定。
def _estimate(item: qec.DecoderEvaluation) -> str:
    rate = item.logical_error_rate
    return f"{rate.value:.4f} [{rate.interval_low:.4f}, {rate.interval_high:.4f}]"


print(f"{'L':>2} {'p':>5} | {'RBM P_fail':>24} {'timeouts':>8} | {'MWPM P_fail':>24} | gate")
for (distance, rate), (scientific, acceptance, _, _) in RESULTS.items():
    rbm, mwpm = scientific.decoder_results
    print(
        f"{distance:>2} {rate:>5.2f} | {_estimate(rbm):>24} {rbm.timeout_count:>8} | "
        f"{_estimate(mwpm):>24} | {acceptance.decision.value.upper()}"
    )

## 4. 图 3：逻辑失败概率与物理错误率

线为 MWPM，带误差棒的标记为 RBM 神经解码器（NeD），颜色区分 L。论文的结论是：阈值（p ≈ 0.109）以下两者几乎相同，阈值以上 NeD 明显更差。

In [ ]:
from IPython.display import Image, display

display(Image(data=qec.figure_png(qec.plot_failure_rate_sweep(POINTS))))

## 5. 图 4：神经解码器返回的同调类

对每个接受的恢复链 `r`，`e ⊕ r` 是闭合环；h0 为可收缩环（成功），h1、h2、h3 分别对应 `Z_L^(1)`、`Z_L^(2)` 和二者之积（逻辑失败）。首个兼容链的选择规则使 NeD 从包含所有同调类的分布中采样，因此随 p 增大非平凡类比例上升。

In [ ]:
FIGURE4_RATES = (0.05, 0.08, 0.12, 0.15)
FIGURE4_POINTS = [point for point in POINTS if point.distance == 4 and point.physical_error_rate in FIGURE4_RATES]
display(Image(data=qec.figure_png(qec.plot_logical_class_histograms(FIGURE4_POINTS or POINTS[:4]))))

## 6. 单点证据

参考点 L=4、p=0.08 的训练监控曲线、同一 test shots 上的失败率比较与同调类分布，均由 `run.visualize` 写入该 Attempt 的 `figures/` 并登记为 Artifact。Gate 的 rationale 给出失败率比值、`LER_RBM − 1.15·LER_MWPM` 的点估计与区间，以及相对容差。

In [ ]:
reference = (4, 0.08) if (4, 0.08) in RESULTS else next(iter(RESULTS))
scientific, acceptance, performance, figures = RESULTS[reference]
print(acceptance.rationale)
print("performance report:", None if performance is None else RUNTIME.artifact_path(performance))
for figure in figures:
    display(Image(filename=str(RUNTIME.artifact_path(figure))))

## 结论与局限

- **能说明什么。** 同一 test shots 上的 NeD 与 MWPM 失败率及其区间、每个点的同调类分布，以及预注册 Gate 的判定，都是可复核的 Artifact（`runs/<experiment>/artifacts/*.json` 记录 checksum 与生产 Attempt）。在 `paper` profile 下，它们复现论文的定性结论：阈值以下 NeD 与 MWPM 接近，阈值以上 NeD 更差；非平凡同调类的比例随 p 上升。
- **不能说明什么。** 超参数按 L 固定而非逐 p 搜索，test 规模为 10⁴，因此逐点数值不承诺与论文一致；Gate 是项目约定，不是论文的判据；吞吐量是批量软件测量，不是实时延迟研究，也不用于比较解码器的计算效率（论文同样指出截止步数使其计算代价不确定）。
- **Gate 容差是事后重新登记的。** 相对 15% 是在看到绝对 0.02 的判定之后选定的，因此不是独立于数据的预注册；两套判定都保留为 Artifact，可以对照。相对容差在低 p 反而更严：L=6、p=0.05 的 RBM 失败率比 MWPM 高约 12%（431 次失败中有 88 次是超时），区间上界超过 15%，判定为 FAIL，而在绝对 0.02 下为 PASS。今后需要更稳定的判定时应增加 test shots，而不是再次调整容差。
- **超时。** RBM 在 `max_steps` 内找不到兼容链的样本计为逻辑失败并在汇总表单独列出；L=6、高 p 时超时更多，这是论文所说的“在合理截止时间内找到兼容链”的瓶颈。